# Complete Machine Learning Pipeline with Support Vector Machine (SVM)

This notebook implements a complete `scikit-learn` numerical pipeline for binary classification on a 100k-row dataset. It includes preprocessing, model training via `SGDClassifier` (Linear SVM), evaluation metrics, and visual representations of the model performance.

In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import (
    classification_report, 
    accuracy_score, 
    confusion_matrix, 
    roc_curve, 
    auc, 
    precision_recall_curve
)

# Set plot style
sns.set_theme(style='whitegrid')
print('Libraries imported successfully.')

## 1. Load Dataset

We load the generated dataset (`svm_dataset_100k.csv`) containing 100,000 samples and 20 numerical features.

In [ ]:
# Load CSV dataset
df = pd.read_csv('svm_dataset_100k.csv')
print(f'Dataset Shape: {df.shape}')
df.head()

## 2. Train / Test Split

Separate the features (`X`) and target variable (`y`), followed by an 80/20 train-test split.

In [ ]:
X = df.drop(columns=['target'])
y = df['target']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Training samples: {X_train.shape[0]}')
print(f'Testing samples:  {X_test.shape[0]}')

## 3. Construct Numerical Pipeline & Train SVM Model

We encapsulate preprocessing (`StandardScaler`) and classifier (`SGDClassifier(loss="hinge")`) into a single `Pipeline`. Using SGD loss allows fast training for large datasets.

In [ ]:
# Define Numerical Pipeline
svm_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('svm', SGDClassifier(loss='hinge', max_iter=1000, tol=1e-3, random_state=42))
])

# Measure training time
start_time = time.time()
svm_pipeline.fit(X_train, y_train)
training_time = time.time() - start_time

print(f'Numerical Pipeline successfully trained in {training_time:.4f} seconds.')

## 4. Evaluation Metrics

Generate standard evaluation metrics including accuracy, precision, recall, and decision margins.

In [ ]:
y_pred = svm_pipeline.predict(X_test)
decision_scores = svm_pipeline.decision_function(X_test)

print(f'Overall Accuracy: {accuracy_score(y_test, y_pred) * 100:.2f}%\n')
print('Classification Report:')
print(classification_report(y_test, y_pred))

## 5. Visual Representations

Below are visual representations evaluating the performance of the trained SVM model.

In [ ]:
# Create figure layout
fig, axes = plt.subplots(2, 2, figsize=(14, 11))

# 1. Confusion Matrix Heatmap
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0, 0], cbar=False)
axes[0, 0].set_title('Confusion Matrix', fontsize=12, fontweight='bold')
axes[0, 0].set_xlabel('Predicted Label')
axes[0, 0].set_ylabel('True Label')
axes[0, 0].set_xticklabels(['Class 0', 'Class 1'])
axes[0, 0].set_yticklabels(['Class 0', 'Class 1'])

# 2. ROC Curve
fpr, tpr, _ = roc_curve(y_test, decision_scores)
roc_auc = auc(fpr, tpr)
axes[0, 1].plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {roc_auc:.3f})')
axes[0, 1].plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
axes[0, 1].set_title('Receiver Operating Characteristic (ROC)', fontsize=12, fontweight='bold')
axes[0, 1].set_xlabel('False Positive Rate')
axes[0, 1].set_ylabel('True Positive Rate')
axes[0, 1].legend(loc='lower right')

# 3. Decision Score Distribution
sns.histplot(decision_scores[y_test == 0], color='blue', label='Class 0', kde=True, ax=axes[1, 0], stat='density', alpha=0.4)
sns.histplot(decision_scores[y_test == 1], color='red', label='Class 1', kde=True, ax=axes[1, 0], stat='density', alpha=0.4)
axes[1, 0].axvline(0, color='black', linestyle='--', label='Decision Boundary (0)')
axes[1, 0].set_title('SVM Decision Function Score Distribution', fontsize=12, fontweight='bold')
axes[1, 0].set_xlabel('Decision Function Score')
axes[1, 0].set_ylabel('Density')
axes[1, 0].legend()

# 4. Feature Weights (Top Coefficients)
coefficients = svm_pipeline.named_steps['svm'].coef_[0]
feature_names = X.columns
sorted_idx = np.argsort(np.abs(coefficients))[::-1]

sns.barplot(x=coefficients[sorted_idx[:10]], y=feature_names[sorted_idx[:10]], palette='vlag', ax=axes[1, 1])
axes[1, 1].set_title('Top 10 Feature Importance (SVM Weights)', fontsize=12, fontweight='bold')
axes[1, 1].set_xlabel('Coefficient Value')
axes[1, 1].set_ylabel('Feature')

plt.tight_layout()
plt.show()